# Silver Zone

## 1. Validación de datos

Validar la integridad referencial de los datos.



Validación ventas de productos inexistentes:

In [0]:
%sql
SELECT
    f.sku AS prod_inexistente,count(*) as count_inexistente
FROM workspace.bronze.raw_fact as f
LEFT JOIN workspace.bronze.raw_producto as p
ON f.sku = p.id_producto
WHERE p.id_producto IS NULL
GROUP BY f.sku
ORDER BY count_inexistente DESC;

Ventas con vendedores inexistentes

In [0]:
%sql
SELECT f.vendedor, COUNT(*) as count_vendedor
FROM workspace.bronze.raw_fact as f
LEFT JOIN workspace.bronze.raw_empleados as e
ON f.vendedor = e.id_vendedor
WHERE e.id_vendedor IS NULL
GROUP BY f.vendedor
ORDER BY count_vendedor DESC;

Sucursales inexistentes en ventas

In [0]:
%sql
SELECT l.id_sucursal, COUNT(*) AS count_sucursal
FROM workspace.bronze.raw_locales as l
LEFT JOIN workspace.bronze.raw_empleados as e
WHERE l.id_sucursal = e.sucursal IS NULL
GROUP BY l.id_sucursal
ORDER BY count_sucursal DESC;

##2. Esquema y tablas

Crear un esquema llamado **"silver"** en el catálogo **"workspace"** y crear dos tablas delta llamadas:


  a) **dim_vendedor** - Que unifique la información de vendedor con la de locales. La tabla debe tener las columnas:


       i. Id_vendedor

       ii. vendedor_nombre

       iii. sucursal_nombre

       iv. region

  b) **dim_producto** - respetando la estructura la tabla equivalente (raw_productos).


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

LOCAL

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.dim_sucursal
USING DELTA AS 
SELECT

    CAST(id_sucursal AS STRING) AS id_sucursal,
    Nombre AS nombre_sucursal,
    tipo AS tipo_sucursal,
    ingestion_date,
    source_file,
    current_timestamp() AS ing_date

FROM workspace.bronze.raw_locales
WHERE

    id_sucursal IS NOT NULL
    AND nombre IS NOT NULL
    AND tipo IS NOT NULL;
    


**STAR** **WH**

dim_vendedor

Debido a que no tenemos ninguna referencia sobre region, se omite esa columna

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.dim_vendedor USING DELTA AS

SELECT 
    CAST(e.id_vendedor AS INT) AS Id_vendedor,
    e.nombre AS vendedor_nombre,
    l.nombre AS sucursal_nombre,
    current_date() AS ing_date
    

FROM workspace.bronze.raw_empleados AS e
INNER JOIN workspace.bronze.raw_locales AS l
ON e.sucursal = l.id_sucursal
WHERE e.id_vendedor IS NOT NULL AND l.id_sucursal IS NOT NULL
AND l.nombre IS NOT NULL AND e.nombre IS NOT NULL



In [0]:
%sql
SELECT * FROM workspace.silver.dim_vendedor

dim_prod

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.dim_producto USING DELTA
AS SELECT DISTINCT
    CAST(id_producto AS INT) AS id_producto,
    familia,
    nombre,
    CAST(precio_unitario AS DOUBLE) AS precio_unitario,
    current_date() AS ing_date
FROM workspace.bronze.raw_producto 
WHERE id_producto IS NOT NULL
AND familia IS NOT NULL
AND nombre IS NOT NULL
AND precio_unitario IS NOT NULL
AND CAST(precio_unitario AS DOUBLE) > 0;
    


In [0]:
%sql
SELECT * FROM workspace.silver.dim_producto


## 3. Tabla de hechos

Crear una tabla delta llamada **fact_ventas** asegurando: 

a) Integridad referencial entre la Fact y las dimensiones.

b) Separar la fecha en los campos: dia, mes, ano

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.fact_venta USING DELTA
AS SELECT 
    CAST(f.sku AS INT) AS id_producto,
    CAST(f.vendedor AS STRING) AS id_vendedor,
    CAST(f.cantidad AS INT) AS cantidad,
    DAY(TO_DATE(f.timestamp, 'yyyy-MM-dd')) AS dia,
    MONTH(TO_DATE(f.timestamp, 'yyyy-MM-dd')) AS mes,
    YEAR(TO_DATE(f.timestamp, 'yyyy-MM-dd')) AS ano,
    current_date() AS ing_date

FROM workspace.bronze.raw_fact f
INNER JOIN workspace.silver.dim_producto p
ON CAST(f.sku AS INT) = p.id_producto
INNER JOIN workspace.silver.dim_vendedor v
ON CAST(f.vendedor AS INT) = v.id_vendedor

WHERE f.sku IS NOT NULL
AND f.vendedor IS NOT NULL
AND f.cantidad IS NOT NULL
AND CAST(f.cantidad AS INT) > 0
AND f.timestamp IS NOT NULL;



In [0]:
%sql
SELECT * FROM workspace.silver.fact_venta